# DR Image Augmentation Pipeline
Complete pipeline: Categorize → Rename → Histogram Matching → NST → DiffuseMix → Frequency-Domain Blending → Supervised Triplet Training → Evaluation

**Prerequisites:** `Messidor_Original/messidor_data.csv`, `Messidor_Original/messidor-2/` (source images), `painters_by_numbers/` (style images), `result/fractal/` (fractal pool)

## 1. Categorize Messidor-2 Images by DR Grade

In [ ]:
import os
import csv
import shutil

csv_path = './Messidor_Original/messidor_data.csv'
source_dir = './Messidor_Original/messidor-2'
output_base = './Messidor_Classified'

grade_names = {
    0: 'Grade_0_No_DR',
    1: 'Grade_1_Mild',
    2: 'Grade_2_Moderate',
    3: 'Grade_3_Severe',
    4: 'Grade_4_Proliferative'
}

for grade, folder_name in grade_names.items():
    os.makedirs(os.path.join(output_base, folder_name), exist_ok=True)

counts = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}
missing = 0

with open(csv_path, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        id_code = row['id_code'].strip()
        diagnosis = int(row['diagnosis'].strip())
        src_path = os.path.join(source_dir, id_code)
        dst_folder = os.path.join(output_base, grade_names[diagnosis])
        dst_path = os.path.join(dst_folder, id_code)
        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)
            counts[diagnosis] += 1
        else:
            missing += 1

print('=' * 50)
print('MESSIDOR-2 CLASSIFICATION SUMMARY')
print('=' * 50)
total = sum(counts.values())
for grade in sorted(counts.keys()):
    pct = (counts[grade] / total * 100) if total > 0 else 0
    print(f'  {grade_names[grade]:<30} : {counts[grade]:>5} images ({pct:.1f}%)')
print(f'  {"Total classified":<30} : {total:>5} images')
print('=' * 50)

## 2. Rename Images to c1.jpg Format

In [ ]:
import os
import csv
from PIL import Image

output_base = './Messidor_Classified'
grade_folders = ['Grade_0_No_DR', 'Grade_1_Mild', 'Grade_2_Moderate', 'Grade_3_Severe', 'Grade_4_Proliferative']

mapping_records = []

for folder_name in grade_folders:
    folder_path = os.path.join(output_base, folder_name)
    if not os.path.exists(folder_path):
        print(f'Skipping {folder_name}')
        continue
    png_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.png')])
    for idx, original_name in enumerate(png_files, start=1):
        src_path = os.path.join(folder_path, original_name)
        new_name = f'c{idx}.jpg'
        dst_path = os.path.join(folder_path, new_name)
        img = Image.open(src_path).convert('RGB')
        img.save(dst_path, 'JPEG', quality=95)
        os.remove(src_path)
        mapping_records.append((folder_name, original_name, new_name))
    print(f'  {folder_name}: {len(png_files)} images renamed')

mapping_path = os.path.join(output_base, 'image_mapping.csv')
with open(mapping_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['folder', 'original_name', 'new_name'])
    writer.writerows(mapping_records)
print(f'\nMapping saved to: {mapping_path}')

## 3. Color Histogram Matching

In [ ]:
import cv2
import os
import csv
import numpy as np
from tqdm import tqdm

class ColorDescriptor:
    def __init__(self, bins=(8, 8, 8)):
        self.bins = bins
    def describe(self, image):
        hist = cv2.calcHist([image], [0, 1, 2], None, self.bins, [0, 180, 0, 256, 0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        return hist.tolist()

def extract_features(dataset_path, output_csv):
    cd = ColorDescriptor()
    with open(output_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        for image_name in tqdm(sorted(os.listdir(dataset_path)), desc=f'Indexing {output_csv}'):
            if not image_name.endswith(('.jpg', '.png', '.jpeg')):
                continue
            image_path = os.path.join(dataset_path, image_name)
            image = cv2.imread(image_path)
            if image is None:
                continue
            features = cd.describe(image)
            writer.writerow([image_path] + features)

def chi2_distance(histA, histB, eps=1e-10):
    return 0.5 * np.sum([((a - b) ** 2) / (a + b + eps) for a, b in zip(histA, histB)])

def match_images(messidor_csv, ptrbynum_csv, output_file):
    painters_data = []
    with open(ptrbynum_csv, 'r') as f:
        for row in csv.reader(f):
            painters_data.append((row[0], [float(x) for x in row[1:]]))
    matches = []
    with open(messidor_csv, 'r') as f:
        for row in tqdm(csv.reader(f), desc=f'Matching {messidor_csv}'):
            messidor_img = row[0]
            messidor_feat = [float(x) for x in row[1:]]
            best_score = float('inf')
            best_match = None
            for painter_img, painter_feat in painters_data:
                d = chi2_distance(messidor_feat, painter_feat)
                if d < best_score:
                    best_score = d
                    best_match = painter_img
            matches.append((messidor_img, best_match))
    with open(output_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Messidor Image', 'Painter Match'])
        writer.writerows(matches)

classified_base = './Messidor_Classified'
ptrbynum_dir = './painters_by_numbers'
grade_folders = ['Grade_0_No_DR', 'Grade_1_Mild', 'Grade_2_Moderate', 'Grade_3_Severe', 'Grade_4_Proliferative']

ptrbynum_csv = os.path.join(classified_base, 'ptrbynum_histogram.csv')
if not os.path.exists(ptrbynum_csv):
    print('Extracting painters_by_numbers histograms...')
    extract_features(ptrbynum_dir, ptrbynum_csv)
else:
    print(f'Using existing {ptrbynum_csv}')

for folder_name in grade_folders:
    grade_dir = os.path.join(classified_base, folder_name)
    if not os.path.exists(grade_dir):
        continue
    img_count = len([f for f in os.listdir(grade_dir) if f.endswith('.jpg')])
    if img_count == 0:
        continue
    print(f'\nProcessing {folder_name} ({img_count} images)')
    histogram_csv = os.path.join(classified_base, f'{folder_name}_histogram.csv')
    extract_features(grade_dir, histogram_csv)
    matches_csv = os.path.join(classified_base, f'{folder_name}_matches.csv')
    match_images(histogram_csv, ptrbynum_csv, matches_csv)
    print(f'  Matches: {matches_csv}')

print('\nAll grades processed!')

## 4. Neural Style Transfer (NST) — Prerequisites

In [ ]:
import os
import numpy as np
import cv2 as cv
import torch
import torch.nn as nn
from torchvision import transforms, models
from torch.optim import LBFGS
from torch.autograd import Variable

IMAGENET_MEAN_255 = [123.675, 116.28, 103.53]
IMAGENET_STD_NEUTRAL = [1, 1, 1]

def load_image(img_path, target_shape=None):
    if not os.path.exists(img_path):
        raise Exception(f'Path not found: {img_path}')
    img = cv.imread(img_path)[:, :, ::-1]
    if target_shape is not None:
        if isinstance(target_shape, int) and target_shape != -1:
            h, w = img.shape[:2]
            new_h = target_shape
            new_w = int(w * (new_h / h))
            img = cv.resize(img, (new_w, new_h), interpolation=cv.INTER_CUBIC)
        else:
            img = cv.resize(img, (target_shape[1], target_shape[0]), interpolation=cv.INTER_CUBIC)
    img = img.astype(np.float32) / 255.0
    return img

def prepare_img(img_path, target_shape, device):
    img = load_image(img_path, target_shape=target_shape)
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.mul(255)),
        transforms.Normalize(mean=IMAGENET_MEAN_255, std=IMAGENET_STD_NEUTRAL)
    ])
    return transform(img).to(device).unsqueeze(0)

def save_image(img_tensor, path):
    img = img_tensor.squeeze().cpu().detach().numpy()
    img = np.moveaxis(img, 0, 2)
    img += np.array(IMAGENET_MEAN_255).reshape((1, 1, 3))
    img = np.clip(img, 0, 255).astype('uint8')
    cv.imwrite(path, img[:, :, ::-1])

def gram_matrix(x, normalize=True):
    (b, ch, h, w) = x.size()
    features = x.view(b, ch, w * h)
    features_t = features.transpose(1, 2)
    gram = features.bmm(features_t)
    if normalize:
        gram /= ch * h * w
    return gram

def total_variation(y):
    return torch.sum(torch.abs(y[:, :, :, :-1] - y[:, :, :, 1:])) + torch.sum(torch.abs(y[:, :, :-1, :] - y[:, :, 1:, :]))

class Vgg19(nn.Module):
    def __init__(self, requires_grad=False):
        super(Vgg19, self).__init__()
        vgg_pretrained_features = models.vgg19(weights='VGG19_Weights.DEFAULT').features
        self.slice = nn.ModuleList()
        self.layer_names = []
        count = 0
        for layer in vgg_pretrained_features:
            if isinstance(layer, nn.ReLU):
                layer = nn.ReLU(inplace=False)
            self.slice.append(layer)
            self.layer_names.append(f'layer_{count}')
            count += 1
        if not requires_grad:
            for param in self.parameters():
                param.requires_grad = False
        self.content_feature_maps_index = 21
        self.style_feature_maps_indices = [0, 5, 10, 19, 28]

    def forward(self, x):
        features = []
        for i, layer in enumerate(self.slice):
            x = layer(x)
            if i in self.style_feature_maps_indices or i == self.content_feature_maps_index:
                features.append(x)
        return features

def neural_style_transfer(config):
    content_img_path = os.path.join(config['content_images_dir'], config['content_img_name'])
    style_img_path = os.path.join(config['style_images_dir'], config['style_img_name'])
    dump_path = config['output_img_dir']
    os.makedirs(dump_path, exist_ok=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    content_img = prepare_img(content_img_path, config['height'], device)
    style_img = prepare_img(style_img_path, config['height'], device)
    init_img = content_img.clone()
    model = Vgg19().to(device).eval()

    content_and_style_feats = model(content_img)
    style_feats = model(style_img)
    content_target = content_and_style_feats[-1].squeeze()
    style_targets = [gram_matrix(feat) for feat in style_feats[:-1]]

    optimizing_img = Variable(init_img.clone(), requires_grad=True)
    optimizer = LBFGS([optimizing_img], max_iter=300)

    iteration = [0]
    def closure():
        optimizer.zero_grad()
        feats = model(optimizing_img)
        content_feat = feats[-1]
        style_feats_out = feats[:-1]
        content_loss = nn.MSELoss()(content_feat.squeeze(), content_target)
        style_loss = 0.0
        for gt, pred in zip(style_targets, [gram_matrix(f) for f in style_feats_out]):
            style_loss += nn.MSELoss()(gt, pred)
        style_loss /= len(style_targets)
        tv_loss = total_variation(optimizing_img)
        total = config['content_weight'] * content_loss + config['style_weight'] * style_loss + config['tv_weight'] * tv_loss
        total.backward()
        if iteration[0] % 50 == 0:
            print(f'Iter {iteration[0]} | Total: {total.item():.2f} | Content: {content_loss.item():.2f} | Style: {style_loss.item():.2f} | TV: {tv_loss.item():.2f}')
        iteration[0] += 1
        return total

    optimizer.step(closure)
    final_img_path = os.path.join(dump_path, f"{os.path.splitext(config['content_img_name'])[0]}_nst.jpg")
    save_image(optimizing_img, final_img_path)
    print(f'Saved to: {final_img_path}')

print('NST functions loaded.')

## 5. Run NST for All Grade Folders

In [ ]:
import os
import pandas as pd

classified_base = './Messidor_Classified'
painters_dir = './painters_by_numbers'
grade_folders = ['Grade_0_No_DR', 'Grade_1_Mild', 'Grade_2_Moderate', 'Grade_3_Severe', 'Grade_4_Proliferative']

for folder_name in grade_folders:
    grade_dir = os.path.join(classified_base, folder_name)
    matches_csv = os.path.join(classified_base, f'{folder_name}_matches.csv')
    nst_output_dir = os.path.join(classified_base, f'{folder_name}_nst')
    if not os.path.exists(matches_csv):
        print(f'Skipping {folder_name} — no matches file')
        continue
    os.makedirs(nst_output_dir, exist_ok=True)
    matches = pd.read_csv(matches_csv)
    total = len(matches)
    print(f'\n{"="*60}\nNST for {folder_name} ({total} images)\n{"="*60}')
    for idx, row in matches.iterrows():
        content_img = os.path.basename(row['Messidor Image'])
        style_img = os.path.basename(row['Painter Match'])
        out_name = f"{os.path.splitext(content_img)[0]}_nst.jpg"
        if os.path.exists(os.path.join(nst_output_dir, out_name)):
            continue
        config = {
            'content_img_name': content_img, 'style_img_name': style_img,
            'height': 400, 'content_weight': 100000.0, 'style_weight': 30000.0, 'tv_weight': 1.0,
            'content_images_dir': grade_dir, 'style_images_dir': painters_dir, 'output_img_dir': nst_output_dir
        }
        try:
            neural_style_transfer(config)
        except Exception as e:
            print(f'  ERROR: {e} — skipping {content_img}')
    done = len([f for f in os.listdir(nst_output_dir) if f.endswith('_nst.jpg')])
    print(f'  Completed: {done}/{total}')

print('\nNST pipeline complete for all grades!')

## 6. DiffuseMix Augmentation (Concatenation + Fractal Blending)

In [ ]:
import os
import random
import numpy as np
from PIL import Image

base_dir = './Messidor_Classified'
grade_folders = ['Grade_0_No_DR', 'Grade_1_Mild', 'Grade_2_Moderate', 'Grade_3_Severe', 'Grade_4_Proliferative']
ALPHA = 0.25

fractal_pool_dir = './result/fractal'
fractal_paths = [os.path.join(fractal_pool_dir, f) for f in os.listdir(fractal_pool_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
fractal_imgs = [Image.open(p).convert('RGB').resize((256, 256)) for p in fractal_paths]
print(f'Loaded {len(fractal_imgs)} fractal images\n')

def combine_images(original_img, augmented_img, blend_width=20):
    width, height = original_img.size
    combine_choice = random.choice(['horizontal', 'vertical'])
    if combine_choice == 'vertical':
        mask = np.linspace(0, 1, blend_width).reshape(-1, 1)
        mask = np.tile(mask, (1, width))
        mask = np.vstack([np.zeros((height // 2 - blend_width // 2, width)), mask, np.ones((height // 2 - blend_width // 2 + blend_width % 2, width))])
        mask = np.tile(mask[:, :, np.newaxis], (1, 1, 3))
    else:
        mask = np.linspace(0, 1, blend_width).reshape(1, -1)
        mask = np.tile(mask, (height, 1))
        mask = np.hstack([np.zeros((height, width // 2 - blend_width // 2)), mask, np.ones((height, width // 2 - blend_width // 2 + blend_width % 2))])
        mask = np.tile(mask[:, :, np.newaxis], (1, 1, 3))
    original_array = np.array(original_img, dtype=np.float32) / 255.0
    augmented_array = np.array(augmented_img, dtype=np.float32) / 255.0
    blended_array = (1 - mask) * original_array + mask * augmented_array
    return Image.fromarray(np.clip(blended_array * 255, 0, 255).astype(np.uint8))

def blend_images_with_resize(base_img, overlay_img, alpha=0.25):
    overlay_resized = overlay_img.resize(base_img.size)
    base_arr = np.array(base_img, dtype=np.float32)
    overlay_arr = np.array(overlay_resized, dtype=np.float32)
    return Image.fromarray(np.clip((1 - alpha) * base_arr + alpha * overlay_arr, 0, 255).astype(np.uint8))

random.seed(42)
for grade in grade_folders:
    orig_dir = os.path.join(base_dir, grade)
    nst_dir = os.path.join(base_dir, f'{grade}_nst')
    concat_dir = os.path.join(base_dir, f'{grade}_concatenated')
    fractal_out_dir = os.path.join(base_dir, f'{grade}_fractal')
    blended_dir = os.path.join(base_dir, f'{grade}_blended')
    os.makedirs(concat_dir, exist_ok=True)
    os.makedirs(fractal_out_dir, exist_ok=True)
    os.makedirs(blended_dir, exist_ok=True)
    orig_images = sorted([f for f in os.listdir(orig_dir) if f.startswith('c') and f.endswith('.jpg')], key=lambda x: int(x[1:].split('.')[0]))
    count = 0
    for img_name in orig_images:
        base_name = os.path.splitext(img_name)[0]
        nst_path = os.path.join(nst_dir, f'{base_name}_nst.jpg')
        if not os.path.exists(nst_path):
            continue
        orig_img = Image.open(os.path.join(orig_dir, img_name)).convert('RGB').resize((256, 256))
        nst_img = Image.open(nst_path).convert('RGB').resize((256, 256))
        concatenated = combine_images(orig_img, nst_img)
        concatenated.save(os.path.join(concat_dir, f'{base_name}_concat.jpg'))
        fractal_img = random.choice(fractal_imgs)
        fractal_img.save(os.path.join(fractal_out_dir, f'{base_name}_fractal.jpg'))
        blended = blend_images_with_resize(concatenated, fractal_img, alpha=ALPHA)
        blended.save(os.path.join(blended_dir, f'{base_name}_blended.jpg'))
        count += 1
    print(f'{grade}: {count} images processed')

print('\nDiffuseMix augmentation complete!')

## 7. Generate Pixel-Blend Variants (6 Alpha Values)

In [ ]:
import os
import numpy as np
from PIL import Image

base_dir = './Messidor_Classified'
grade_folders = ['Grade_0_No_DR', 'Grade_1_Mild', 'Grade_2_Moderate', 'Grade_3_Severe', 'Grade_4_Proliferative']
alphas = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

def blend_images(base_img, overlay_img, alpha):
    overlay_resized = overlay_img.resize(base_img.size)
    base_arr = np.array(base_img, dtype=np.float32)
    overlay_arr = np.array(overlay_resized, dtype=np.float32)
    return Image.fromarray(np.clip((1 - alpha) * base_arr + alpha * overlay_arr, 0, 255).astype(np.uint8))

for alpha in alphas:
    alpha_str = f'{int(alpha*100):03d}'
    count = 0
    for grade in grade_folders:
        concat_dir = os.path.join(base_dir, f'{grade}_concatenated')
        fractal_dir = os.path.join(base_dir, f'{grade}_fractal')
        output_dir = os.path.join(base_dir, f'{grade}_blended_{alpha_str}')
        os.makedirs(output_dir, exist_ok=True)
        for cf in sorted(os.listdir(concat_dir)):
            if not cf.endswith('_concat.jpg'): continue
            base_name = cf.replace('_concat.jpg', '')
            fractal_path = os.path.join(fractal_dir, f'{base_name}_fractal.jpg')
            if not os.path.exists(fractal_path): continue
            concat_img = Image.open(os.path.join(concat_dir, cf)).convert('RGB')
            fractal_img = Image.open(fractal_path).convert('RGB')
            blended = blend_images(concat_img, fractal_img, alpha)
            blended.save(os.path.join(output_dir, f'{base_name}_blended.jpg'))
            count += 1
    print(f'  alpha={alpha:.2f} -> {count} images')

print('\nAll pixel-blend variants generated!')

## 8. Generate Frequency-Domain Blend Variants (6 Alpha Values)

In [ ]:
import os
import numpy as np
from PIL import Image

def frequency_blend(base_img, overlay_img, alpha=0.15, cutoff_ratio=0.3):
    overlay_resized = overlay_img.resize(base_img.size)
    base_arr = np.array(base_img, dtype=np.float64)
    overlay_arr = np.array(overlay_resized, dtype=np.float64)
    h, w = base_arr.shape[:2]
    result = np.zeros_like(base_arr, dtype=np.float64)
    cy, cx = h // 2, w // 2
    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((Y - cy)**2 + (X - cx)**2)
    max_dist = np.sqrt(cy**2 + cx**2)
    sigma = cutoff_ratio * max_dist
    low_pass = np.exp(-(dist**2) / (2 * sigma**2 + 1e-10))
    high_pass = 1.0 - low_pass
    for c in range(3):
        base_fft = np.fft.fftshift(np.fft.fft2(base_arr[:, :, c]))
        overlay_fft = np.fft.fftshift(np.fft.fft2(overlay_arr[:, :, c]))
        blended_fft = low_pass * base_fft + high_pass * ((1 - alpha) * base_fft + alpha * overlay_fft)
        result[:, :, c] = np.real(np.fft.ifft2(np.fft.ifftshift(blended_fft)))
    return Image.fromarray(np.clip(result, 0, 255).astype(np.uint8))

base_dir = './Messidor_Classified'
grade_folders = ['Grade_0_No_DR', 'Grade_1_Mild', 'Grade_2_Moderate', 'Grade_3_Severe', 'Grade_4_Proliferative']
alphas = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
CUTOFF_RATIO = 0.3

for alpha in alphas:
    alpha_str = f'{int(alpha*100):03d}'
    count = 0
    for grade in grade_folders:
        concat_dir = os.path.join(base_dir, f'{grade}_concatenated')
        fractal_dir = os.path.join(base_dir, f'{grade}_fractal')
        output_dir = os.path.join(base_dir, f'{grade}_freqblend_{alpha_str}')
        os.makedirs(output_dir, exist_ok=True)
        for cf in sorted(os.listdir(concat_dir)):
            if not cf.endswith('_concat.jpg'): continue
            base_name = cf.replace('_concat.jpg', '')
            fractal_path = os.path.join(fractal_dir, f'{base_name}_fractal.jpg')
            if not os.path.exists(fractal_path): continue
            concat_img = Image.open(os.path.join(concat_dir, cf)).convert('RGB')
            fractal_img = Image.open(fractal_path).convert('RGB')
            blended = frequency_blend(concat_img, fractal_img, alpha=alpha, cutoff_ratio=CUTOFF_RATIO)
            blended.save(os.path.join(output_dir, f'{base_name}_blended.jpg'))
            count += 1
    print(f'  alpha={alpha:.2f} -> {count} freq-blended images')

print(f'\nFrequency-domain blending complete (cutoff={CUTOFF_RATIO})!')

## 9. Supervised Training Infrastructure
Dataset class (weighted sampling + stratified negatives), model, training loop, and evaluation functions.

In [ ]:
import os, copy, random
import numpy as np
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import resnet18
from tqdm import tqdm

import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score, confusion_matrix, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.manifold import TSNE

# === Dataset ===

class ImprovedSupervisedTripletDataset(Dataset):
    def __init__(self, base_dir, grade_folders, aug_suffix, aug_folder_suffix,
                 transform=None, neg_seed=42):
        self.transform = transform
        self.samples = []
        self.grade_originals = {}
        for grade_idx, grade in enumerate(grade_folders):
            orig_dir = os.path.join(base_dir, grade)
            aug_dir = os.path.join(base_dir, f'{grade}{aug_folder_suffix}')
            grade_orig_paths = []
            for f in sorted(os.listdir(orig_dir)):
                if f.startswith('c') and f.endswith('.jpg'):
                    base_name = os.path.splitext(f)[0]
                    aug_name = f'{base_name}{aug_suffix}'
                    aug_path = os.path.join(aug_dir, aug_name)
                    if os.path.exists(aug_path):
                        anchor_path = os.path.join(orig_dir, f)
                        self.samples.append((anchor_path, aug_path, grade_idx))
                        grade_orig_paths.append(anchor_path)
            self.grade_originals[grade_idx] = grade_orig_paths
        rng = np.random.RandomState(neg_seed)
        self.neg_paths = []
        all_grades = sorted(self.grade_originals.keys())
        for _, _, grade_idx in self.samples:
            other_grades = [g for g in all_grades if g != grade_idx]
            neg_grade = other_grades[len(self.neg_paths) % len(other_grades)]
            neg_idx = rng.randint(0, len(self.grade_originals[neg_grade]))
            self.neg_paths.append(self.grade_originals[neg_grade][neg_idx])
        self.labels = [s[2] for s in self.samples]
        label_counts = Counter(self.labels)
        total = len(self.labels)
        self.sample_weights = [total / label_counts[label] for label in self.labels]
        print(f'  {len(self.samples)} triplets, class dist: {dict(sorted(label_counts.items()))}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        anchor_path, pos_path, _ = self.samples[idx]
        neg_path = self.neg_paths[idx]
        anchor = Image.open(anchor_path).convert('RGB')
        positive = Image.open(pos_path).convert('RGB')
        negative = Image.open(neg_path).convert('RGB')
        if self.transform:
            anchor = self.transform(anchor)
            positive = self.transform(positive)
            negative = self.transform(negative)
        return anchor, positive, negative

# === Model ===

class EmbeddingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.resnet = resnet18(weights='IMAGENET1K_V1')
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, 128)
    def forward(self, x):
        return F.normalize(self.resnet(x), p=2, dim=1)

class TripletLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin
    def forward(self, anchor, positive, negative):
        pos_dist = F.pairwise_distance(anchor, positive, p=2)
        neg_dist = F.pairwise_distance(anchor, negative, p=2)
        return F.relu(pos_dist - neg_dist + self.margin).mean()

# === Training Utilities ===

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def make_weighted_loader(dataset, batch_size=16, seed=42):
    weights = torch.DoubleTensor(dataset.sample_weights)
    sampler = WeightedRandomSampler(weights, num_samples=len(dataset), replacement=True,
                                     generator=torch.Generator().manual_seed(seed))
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                      num_workers=0, pin_memory=torch.cuda.is_available())

def train_with_checkpoints(model, loader, optimizer, loss_fn, total_epochs, checkpoint_every, device):
    model.train()
    checkpoints = {}
    for epoch in range(1, total_epochs + 1):
        total_loss = 0.0
        pbar = tqdm(loader, desc=f'Epoch {epoch}/{total_epochs}')
        for anchor, positive, negative in pbar:
            anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(anchor), model(positive), model(negative))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix(loss=loss.item())
        avg_loss = total_loss / len(loader)
        print(f'Epoch {epoch} - Avg Loss: {avg_loss:.4f}')
        if epoch % checkpoint_every == 0:
            checkpoints[epoch] = copy.deepcopy(model.state_dict())
            print(f'  >> Checkpoint saved at epoch {epoch}')
    return checkpoints

# === Evaluation Utilities ===

def extract_embeddings_with_labels(model, dataset, device):
    model.eval()
    embeddings = []
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
    with torch.no_grad():
        for anchor, _, _ in loader:
            emb = model(anchor.to(device))
            embeddings.append(emb.cpu().numpy())
    return np.vstack(embeddings), np.array(dataset.labels)

def linear_probe_accuracy(embeddings, labels, n_classes=5, epochs=100, lr=0.01):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    all_preds = np.zeros_like(labels)
    for train_idx, val_idx in skf.split(embeddings, labels):
        X_train = torch.FloatTensor(embeddings[train_idx])
        y_train = torch.LongTensor(labels[train_idx])
        X_val = torch.FloatTensor(embeddings[val_idx])
        classifier = nn.Linear(128, n_classes)
        optimizer = torch.optim.Adam(classifier.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()
        classifier.train()
        for _ in range(epochs):
            optimizer.zero_grad()
            loss = loss_fn(classifier(X_train), y_train)
            loss.backward()
            optimizer.step()
        classifier.eval()
        with torch.no_grad():
            all_preds[val_idx] = classifier(X_val).argmax(dim=1).numpy()
    return (all_preds == labels).mean(), all_preds

print('Training infrastructure loaded.')

## 10. Phase 1: Alpha Ablation (Pixel-Blend + Freq-Blend)

In [ ]:
import os, copy, random
import numpy as np
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import resnet18
from tqdm import tqdm

import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

# ── Inline definitions (no Cell 9 dependency) ──

class ImprovedSupervisedTripletDataset(Dataset):
    def __init__(self, base_dir, grade_folders, aug_suffix, aug_folder_suffix,
                 transform=None, neg_seed=42):
        self.transform = transform
        self.samples = []
        self.grade_originals = {}
        for grade_idx, grade in enumerate(grade_folders):
            orig_dir = os.path.join(base_dir, grade)
            aug_dir = os.path.join(base_dir, f'{grade}{aug_folder_suffix}')
            grade_orig_paths = []
            for f in sorted(os.listdir(orig_dir)):
                if f.startswith('c') and f.endswith('.jpg'):
                    base_name = os.path.splitext(f)[0]
                    aug_name = f'{base_name}{aug_suffix}'
                    aug_path = os.path.join(aug_dir, aug_name)
                    if os.path.exists(aug_path):
                        anchor_path = os.path.join(orig_dir, f)
                        self.samples.append((anchor_path, aug_path, grade_idx))
                        grade_orig_paths.append(anchor_path)
            self.grade_originals[grade_idx] = grade_orig_paths
        rng = np.random.RandomState(neg_seed)
        self.neg_paths = []
        all_grades = sorted(self.grade_originals.keys())
        for _, _, grade_idx in self.samples:
            other_grades = [g for g in all_grades if g != grade_idx]
            neg_grade = other_grades[len(self.neg_paths) % len(other_grades)]
            neg_idx = rng.randint(0, len(self.grade_originals[neg_grade]))
            self.neg_paths.append(self.grade_originals[neg_grade][neg_idx])
        self.labels = [s[2] for s in self.samples]
        label_counts = Counter(self.labels)
        total = len(self.labels)
        self.sample_weights = [total / label_counts[label] for label in self.labels]
        print(f'  {len(self.samples)} triplets, class dist: {dict(sorted(label_counts.items()))}')
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        anchor_path, pos_path, _ = self.samples[idx]
        neg_path = self.neg_paths[idx]
        anchor = Image.open(anchor_path).convert('RGB')
        positive = Image.open(pos_path).convert('RGB')
        negative = Image.open(neg_path).convert('RGB')
        if self.transform:
            anchor = self.transform(anchor)
            positive = self.transform(positive)
            negative = self.transform(negative)
        return anchor, positive, negative

class EmbeddingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.resnet = resnet18(weights='IMAGENET1K_V1')
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, 128)
    def forward(self, x):
        return F.normalize(self.resnet(x), p=2, dim=1)

class TripletLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin
    def forward(self, anchor, positive, negative):
        pos_dist = F.pairwise_distance(anchor, positive, p=2)
        neg_dist = F.pairwise_distance(anchor, negative, p=2)
        return F.relu(pos_dist - neg_dist + self.margin).mean()

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def make_weighted_loader(dataset, batch_size=16, seed=42):
    weights = torch.DoubleTensor(dataset.sample_weights)
    sampler = WeightedRandomSampler(weights, num_samples=len(dataset), replacement=True,
                                     generator=torch.Generator().manual_seed(seed))
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                      num_workers=0, pin_memory=torch.cuda.is_available())

def train_with_checkpoints(model, loader, optimizer, loss_fn, total_epochs, checkpoint_every, device):
    model.train()
    checkpoints = {}
    for epoch in range(1, total_epochs + 1):
        total_loss = 0.0
        pbar = tqdm(loader, desc=f'Epoch {epoch}/{total_epochs}')
        for anchor, positive, negative in pbar:
            anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(anchor), model(positive), model(negative))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix(loss=loss.item())
        avg_loss = total_loss / len(loader)
        print(f'Epoch {epoch} - Avg Loss: {avg_loss:.4f}')
        if epoch % checkpoint_every == 0:
            checkpoints[epoch] = copy.deepcopy(model.state_dict())
            print(f'  >> Checkpoint saved at epoch {epoch}')
    return checkpoints

def extract_embeddings_with_labels(model, dataset, device):
    model.eval()
    embeddings = []
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
    with torch.no_grad():
        for anchor, _, _ in loader:
            emb = model(anchor.to(device))
            embeddings.append(emb.cpu().numpy())
    return np.vstack(embeddings), np.array(dataset.labels)

def linear_probe_accuracy(embeddings, labels, n_classes=5, epochs=100, lr=0.01):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    all_preds = np.zeros_like(labels)
    for train_idx, val_idx in skf.split(embeddings, labels):
        X_train = torch.FloatTensor(embeddings[train_idx])
        y_train = torch.LongTensor(labels[train_idx])
        X_val = torch.FloatTensor(embeddings[val_idx])
        classifier = nn.Linear(128, n_classes)
        optimizer = torch.optim.Adam(classifier.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()
        classifier.train()
        for _ in range(epochs):
            optimizer.zero_grad()
            loss = loss_fn(classifier(X_train), y_train)
            loss.backward()
            optimizer.step()
        classifier.eval()
        with torch.no_grad():
            all_preds[val_idx] = classifier(X_val).argmax(dim=1).numpy()
    return (all_preds == labels).mean(), all_preds

# ── Phase 1 Ablation ──

base_dir = './Messidor_Classified'
grade_folders = ['Grade_0_No_DR', 'Grade_1_Mild', 'Grade_2_Moderate', 'Grade_3_Severe', 'Grade_4_Proliferative']
alphas = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
TOTAL_EPOCHS = 75
CHECKPOINT_EVERY = 25
LR = 1e-4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def run_ablation(blend_type, folder_suffix_template):
    print(f'\n{"="*60}\n{blend_type} Ablation\n{"="*60}')
    datasets = {}
    for alpha in alphas:
        alpha_str = f'{int(alpha*100):03d}'
        name = f'{blend_type}_a={alpha:.2f}'
        datasets[name] = ImprovedSupervisedTripletDataset(
            base_dir, grade_folders, '_blended.jpg',
            folder_suffix_template.format(alpha_str), transform, neg_seed=42)

    results = {}
    for name, dataset in datasets.items():
        print(f'\n--- Training {name} ({TOTAL_EPOCHS} epochs) ---')
        seed_everything(42)
        model = EmbeddingNet().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        loss_fn = TripletLoss(margin=1.0)
        loader = make_weighted_loader(dataset, batch_size=16, seed=42)
        checkpoints = train_with_checkpoints(model, loader, optimizer, loss_fn, TOTAL_EPOCHS, CHECKPOINT_EVERY, device)
        
        best_probe = -1
        for epoch in sorted(checkpoints.keys()):
            model = EmbeddingNet().to(device)
            model.load_state_dict(checkpoints[epoch])
            embeddings, labels = extract_embeddings_with_labels(model, dataset, device)
            sil = silhouette_score(embeddings, labels)
            knn = KNeighborsClassifier(n_neighbors=5)
            cv = cross_val_score(knn, embeddings, labels, cv=5, scoring='accuracy')
            probe_acc, _ = linear_probe_accuracy(embeddings, labels)
            if probe_acc > best_probe:
                best_probe, best_knn, best_sil, best_epoch = probe_acc, cv.mean(), sil, epoch
        results[name] = {'best_probe': best_probe, 'best_knn': best_knn, 'best_sil': best_sil, 'best_epoch': best_epoch}
        print(f'  Best: Probe={best_probe:.4f}, kNN={best_knn:.4f}, Sil={best_sil:.4f} @ Epoch {best_epoch}')
    return results

# Run both ablations
pixel_results = run_ablation('Pixel', '_blended_{}')
freq_results = run_ablation('Freq', '_freqblend_{}')

# Find best alphas
best_pixel_name = max(pixel_results, key=lambda k: pixel_results[k]['best_probe'])
best_pixel_alpha = float(best_pixel_name.split('=')[1])
best_freq_name = max(freq_results, key=lambda k: freq_results[k]['best_probe'])
best_freq_alpha = float(best_freq_name.split('=')[1])

print(f'\n{"="*60}')
print(f'BEST PIXEL ALPHA: {best_pixel_alpha:.2f} (Probe={pixel_results[best_pixel_name]["best_probe"]:.4f})')
print(f'BEST FREQ ALPHA:  {best_freq_alpha:.2f} (Probe={freq_results[best_freq_name]["best_probe"]:.4f})')
print(f'{"="*60}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (title, res) in zip(axes, [('Pixel-Blend', pixel_results), ('Freq-Blend', freq_results)]):
    vals = [res[f'{title.split("-")[0]}_a={a:.2f}']['best_probe'] for a in alphas]
    ax.plot(alphas, vals, 'go-', linewidth=2, markersize=8)
    ax.set_xlabel('Alpha'); ax.set_ylabel('Best Linear Probe')
    ax.set_title(f'{title}: Probe vs Alpha'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Phase 2: Final 4-Way Comparison
NST vs Concat vs Pixel-Blend(best alpha) vs Freq-Blend(best alpha)

In [ ]:
import os, copy, random
import numpy as np
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import resnet18
from tqdm import tqdm

import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score, confusion_matrix, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.manifold import TSNE

# ── Inline definitions (no Cell 9 dependency) ──

class ImprovedSupervisedTripletDataset(Dataset):
    def __init__(self, base_dir, grade_folders, aug_suffix, aug_folder_suffix,
                 transform=None, neg_seed=42):
        self.transform = transform
        self.samples = []
        self.grade_originals = {}
        for grade_idx, grade in enumerate(grade_folders):
            orig_dir = os.path.join(base_dir, grade)
            aug_dir = os.path.join(base_dir, f'{grade}{aug_folder_suffix}')
            grade_orig_paths = []
            for f in sorted(os.listdir(orig_dir)):
                if f.startswith('c') and f.endswith('.jpg'):
                    base_name = os.path.splitext(f)[0]
                    aug_name = f'{base_name}{aug_suffix}'
                    aug_path = os.path.join(aug_dir, aug_name)
                    if os.path.exists(aug_path):
                        anchor_path = os.path.join(orig_dir, f)
                        self.samples.append((anchor_path, aug_path, grade_idx))
                        grade_orig_paths.append(anchor_path)
            self.grade_originals[grade_idx] = grade_orig_paths
        rng = np.random.RandomState(neg_seed)
        self.neg_paths = []
        all_grades = sorted(self.grade_originals.keys())
        for _, _, grade_idx in self.samples:
            other_grades = [g for g in all_grades if g != grade_idx]
            neg_grade = other_grades[len(self.neg_paths) % len(other_grades)]
            neg_idx = rng.randint(0, len(self.grade_originals[neg_grade]))
            self.neg_paths.append(self.grade_originals[neg_grade][neg_idx])
        self.labels = [s[2] for s in self.samples]
        label_counts = Counter(self.labels)
        total = len(self.labels)
        self.sample_weights = [total / label_counts[label] for label in self.labels]
        print(f'  {len(self.samples)} triplets, class dist: {dict(sorted(label_counts.items()))}')
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        anchor_path, pos_path, _ = self.samples[idx]
        neg_path = self.neg_paths[idx]
        anchor = Image.open(anchor_path).convert('RGB')
        positive = Image.open(pos_path).convert('RGB')
        negative = Image.open(neg_path).convert('RGB')
        if self.transform:
            anchor = self.transform(anchor)
            positive = self.transform(positive)
            negative = self.transform(negative)
        return anchor, positive, negative

class EmbeddingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.resnet = resnet18(weights='IMAGENET1K_V1')
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, 128)
    def forward(self, x):
        return F.normalize(self.resnet(x), p=2, dim=1)

class TripletLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin
    def forward(self, anchor, positive, negative):
        pos_dist = F.pairwise_distance(anchor, positive, p=2)
        neg_dist = F.pairwise_distance(anchor, negative, p=2)
        return F.relu(pos_dist - neg_dist + self.margin).mean()

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def make_weighted_loader(dataset, batch_size=16, seed=42):
    weights = torch.DoubleTensor(dataset.sample_weights)
    sampler = WeightedRandomSampler(weights, num_samples=len(dataset), replacement=True,
                                     generator=torch.Generator().manual_seed(seed))
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                      num_workers=0, pin_memory=torch.cuda.is_available())

def train_with_checkpoints(model, loader, optimizer, loss_fn, total_epochs, checkpoint_every, device):
    model.train()
    checkpoints = {}
    for epoch in range(1, total_epochs + 1):
        total_loss = 0.0
        pbar = tqdm(loader, desc=f'Epoch {epoch}/{total_epochs}')
        for anchor, positive, negative in pbar:
            anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(anchor), model(positive), model(negative))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix(loss=loss.item())
        avg_loss = total_loss / len(loader)
        print(f'Epoch {epoch} - Avg Loss: {avg_loss:.4f}')
        if epoch % checkpoint_every == 0:
            checkpoints[epoch] = copy.deepcopy(model.state_dict())
            print(f'  >> Checkpoint saved at epoch {epoch}')
    return checkpoints

def extract_embeddings_with_labels(model, dataset, device):
    model.eval()
    embeddings = []
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
    with torch.no_grad():
        for anchor, _, _ in loader:
            emb = model(anchor.to(device))
            embeddings.append(emb.cpu().numpy())
    return np.vstack(embeddings), np.array(dataset.labels)

def linear_probe_accuracy(embeddings, labels, n_classes=5, epochs=100, lr=0.01):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    all_preds = np.zeros_like(labels)
    for train_idx, val_idx in skf.split(embeddings, labels):
        X_train = torch.FloatTensor(embeddings[train_idx])
        y_train = torch.LongTensor(labels[train_idx])
        X_val = torch.FloatTensor(embeddings[val_idx])
        classifier = nn.Linear(128, n_classes)
        optimizer = torch.optim.Adam(classifier.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()
        classifier.train()
        for _ in range(epochs):
            optimizer.zero_grad()
            loss = loss_fn(classifier(X_train), y_train)
            loss.backward()
            optimizer.step()
        classifier.eval()
        with torch.no_grad():
            all_preds[val_idx] = classifier(X_val).argmax(dim=1).numpy()
    return (all_preds == labels).mean(), all_preds

# ── Phase 2: Final 4-Way Comparison ──

base_dir = './Messidor_Classified'
grade_folders = ['Grade_0_No_DR', 'Grade_1_Mild', 'Grade_2_Moderate', 'Grade_3_Severe', 'Grade_4_Proliferative']
grade_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
LR = 1e-4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

best_pixel_str = f'{int(best_pixel_alpha*100):03d}'
best_freq_str = f'{int(best_freq_alpha*100):03d}'
TOTAL_EPOCHS = 125
CHECKPOINT_EVERY = 25

print(f'Phase 2: Pixel best a={best_pixel_alpha:.2f}, Freq best a={best_freq_alpha:.2f}\n')

final_datasets = {
    'NST': ImprovedSupervisedTripletDataset(base_dir, grade_folders, '_nst.jpg', '_nst', transform, neg_seed=42),
    'Concat': ImprovedSupervisedTripletDataset(base_dir, grade_folders, '_concat.jpg', '_concatenated', transform, neg_seed=42),
    f'Pixel(a={best_pixel_alpha:.2f})': ImprovedSupervisedTripletDataset(base_dir, grade_folders, '_blended.jpg', f'_blended_{best_pixel_str}', transform, neg_seed=42),
    f'Freq(a={best_freq_alpha:.2f})': ImprovedSupervisedTripletDataset(base_dir, grade_folders, '_blended.jpg', f'_freqblend_{best_freq_str}', transform, neg_seed=42),
}

final_checkpoints = {}
for name, dataset in final_datasets.items():
    print(f'\n{"="*60}\n=== Training {name} ({TOTAL_EPOCHS} epochs) ===\n{"="*60}')
    seed_everything(42)
    model = EmbeddingNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = TripletLoss(margin=1.0)
    loader = make_weighted_loader(dataset, batch_size=16, seed=42)
    final_checkpoints[name] = train_with_checkpoints(model, loader, optimizer, loss_fn, TOTAL_EPOCHS, CHECKPOINT_EVERY, device)

# === Evaluate ===
final_results = {}
for name in final_checkpoints:
    checkpoints = final_checkpoints[name]
    dataset = final_datasets[name]
    epochs = sorted(checkpoints.keys())
    sil_scores, knn_scores, probe_scores = [], [], []
    print(f'\nEvaluating {name}...')
    for epoch in epochs:
        model = EmbeddingNet().to(device)
        model.load_state_dict(checkpoints[epoch])
        embeddings, labels = extract_embeddings_with_labels(model, dataset, device)
        sil_scores.append(silhouette_score(embeddings, labels))
        knn = KNeighborsClassifier(n_neighbors=5)
        knn_scores.append(cross_val_score(knn, embeddings, labels, cv=5, scoring='accuracy').mean())
        probe_acc, _ = linear_probe_accuracy(embeddings, labels)
        probe_scores.append(probe_acc)
        print(f'  Epoch {epoch}: Sil={sil_scores[-1]:.4f}, kNN={knn_scores[-1]:.4f}, Probe={probe_acc:.4f}')
    final_results[name] = {
        'epochs': epochs, 'silhouette': sil_scores, 'knn_accuracy': knn_scores, 'probe_accuracy': probe_scores,
        'best_sil': max(sil_scores), 'best_sil_epoch': epochs[np.argmax(sil_scores)],
        'best_knn': max(knn_scores), 'best_knn_epoch': epochs[np.argmax(knn_scores)],
        'best_probe': max(probe_scores), 'best_probe_epoch': epochs[np.argmax(probe_scores)],
    }

# === Learning Curves ===
fig, axes = plt.subplots(1, 3, figsize=(21, 5))
for name, r in final_results.items():
    axes[0].plot(r['epochs'], r['silhouette'], 'o-', label=name)
    axes[1].plot(r['epochs'], r['knn_accuracy'], 'o-', label=name)
    axes[2].plot(r['epochs'], r['probe_accuracy'], 'o-', label=name)
axes[0].set_title('Silhouette'); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[1].set_title('kNN Accuracy'); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
axes[2].set_title('Linear Probe'); axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)
plt.suptitle('Phase 2: 4-Way Final Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# === Summary Table ===
print('\n' + '='*110)
print(f'{"Method":<22} {"Best Sil":>10} {"@Ep":>5} {"Best kNN":>10} {"@Ep":>5} {"Best Probe":>12} {"@Ep":>5}')
print('='*110)
for name, r in final_results.items():
    print(f'{name:<22} {r["best_sil"]:>10.4f} {r["best_sil_epoch"]:>5} {r["best_knn"]:>10.4f} {r["best_knn_epoch"]:>5} {r["best_probe"]:>12.4f} {r["best_probe_epoch"]:>5}')
print('='*110)

# === Confusion Matrices ===
fig, axes = plt.subplots(1, len(final_results), figsize=(6*len(final_results), 5.5))
for ax, (name, r) in zip(axes, final_results.items()):
    model = EmbeddingNet().to(device)
    model.load_state_dict(final_checkpoints[name][r['best_probe_epoch']])
    embeddings, labels = extract_embeddings_with_labels(model, final_datasets[name], device)
    _, preds = linear_probe_accuracy(embeddings, labels)
    cm = confusion_matrix(labels, preds)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'{name}\nProbe: {r["best_probe"]:.4f}', fontsize=10)
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    ax.set_xticklabels(grade_names, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(grade_names, fontsize=7)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f'{cm[i,j]}\n({cm_norm[i,j]:.0%})', ha='center', va='center', fontsize=6, color='white' if cm_norm[i,j] > 0.5 else 'black')
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle('Confusion Matrices (Linear Probe)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# === t-SNE ===
fig, axes = plt.subplots(1, len(final_results), figsize=(6*len(final_results), 5.5))
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6']
for ax, (name, r) in zip(axes, final_results.items()):
    model = EmbeddingNet().to(device)
    model.load_state_dict(final_checkpoints[name][r['best_probe_epoch']])
    embeddings, labels = extract_embeddings_with_labels(model, final_datasets[name], device)
    emb_2d = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(embeddings)
    for gi, gn in enumerate(grade_names):
        mask = labels == gi
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], c=colors[gi], label=f'{gn} ({mask.sum()})', alpha=0.6, s=15)
    ax.set_title(f'{name}\nProbe={r["best_probe"]:.4f}', fontsize=10)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.2)
plt.suptitle('t-SNE Embeddings by DR Grade', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# === Classification Reports ===
for name, r in final_results.items():
    model = EmbeddingNet().to(device)
    model.load_state_dict(final_checkpoints[name][r['best_probe_epoch']])
    embeddings, labels = extract_embeddings_with_labels(model, final_datasets[name], device)
    _, preds = linear_probe_accuracy(embeddings, labels)
    print(f'\n--- {name} (Epoch {r["best_probe_epoch"]}) ---')
    print(classification_report(labels, preds, target_names=grade_names, digits=4))

# === Winner ===
best_method = max(final_results, key=lambda k: final_results[k]['best_probe'])
print(f'\n{"="*60}')
print(f'OVERALL WINNER: {best_method}')
print(f'  Probe: {final_results[best_method]["best_probe"]:.4f}')
print(f'  kNN:   {final_results[best_method]["best_knn"]:.4f}')
print(f'  Sil:   {final_results[best_method]["best_sil"]:.4f}')
print(f'{"="*60}')

# Pixel vs Freq
pixel_name = f'Pixel(a={best_pixel_alpha:.2f})'
freq_name = f'Freq(a={best_freq_alpha:.2f})'
if pixel_name in final_results and freq_name in final_results:
    diff = final_results[freq_name]['best_probe'] - final_results[pixel_name]['best_probe']
    print(f'\nFreq vs Pixel: {diff:+.4f} ({"Freq wins" if diff > 0 else "Pixel wins"})')